## データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# THEMIS-Aの電場・磁場データのplot

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/20:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/2230-2330_low_freq'

psp.themis.fgm(trange=time_range, probe='a', level='l2')                    # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
#psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efw')    # efw: 16448 Hz
#psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp')    # efp: 512 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2')                    # eff: 8 Hz

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

E8_data_gsm     = pt.data_quants['tha_eff_dot0_gsm']
B16_data_gsm    = pt.data_quants['tha_fgl_gsm']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E8_data_gsm   = E8_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B16_data_gsm   = B16_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

print(E8_data_gsm)
print('')
print(B16_data_gsm)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

E8_data_gsm_x = E8_data_gsm.isel(v_dim=0)
E8_data_gsm_y = E8_data_gsm.isel(v_dim=1)
E8_data_gsm_z = E8_data_gsm.isel(v_dim=2)

B16_data_gsm_x = B16_data_gsm.isel(v_dim=0)
B16_data_gsm_y = B16_data_gsm.isel(v_dim=1)
B16_data_gsm_z = B16_data_gsm.isel(v_dim=2)

pt.store_data('E8_gsm_x', data={'x': E8_data_gsm_x.time, 'y': E8_data_gsm_x})
pt.store_data('E8_gsm_y', data={'x': E8_data_gsm_x.time, 'y': E8_data_gsm_x})
pt.store_data('E8_gsm_z', data={'x': E8_data_gsm_x.time, 'y': E8_data_gsm_x})
pt.store_data('B16_gsm_x', data={'x': B16_data_gsm_x.time, 'y': B16_data_gsm_x})
pt.store_data('B16_gsm_y', data={'x': B16_data_gsm_y.time, 'y': B16_data_gsm_y})
pt.store_data('B16_gsm_z', data={'x': B16_data_gsm_z.time, 'y': B16_data_gsm_z})

pt.options('E8_gsm_x', 'ytitle', 'E_x (gsm)')
pt.options('E8_gsm_y', 'ytitle', 'E_y (gsm)')
pt.options('E8_gsm_z', 'ytitle', 'E_z (gsm)')
pt.options(['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z'], 'ysubtitle', '[mV/m]')
pt.options('B16_gsm_x', 'ytitle', 'B_x (gsm)')
pt.options('B16_gsm_y', 'ytitle', 'B_y (gsm)')
pt.options('B16_gsm_z', 'ytitle', 'B_z (gsm)')
pt.options(['B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z'], 'ysubtitle', '[nT]')

pt.timespan('2022-09-01/20:00:00', 3, keyword='hour')
pt.tplot(['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z'])

pt.timespan('2022-09-01/20:45:00', 95, keyword='minute')
pt.tplot(['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z'])

pt.timespan('2022-09-01/22:20:00', 1, keyword='hour')
pt.tplot(['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z'])

In [ ]:
import numpy as np
import pytplot as pt
import matplotlib.pyplot as plt

gap_thr = np.timedelta64(30, 's')

# ---------- 1. 電場チャンク ----------------------------------
t_ref_E = pt.data_quants['E8_gsm_x'].time.values
is_new_E = np.concatenate(([True], np.diff(t_ref_E) > gap_thr))
chunk_E  = np.cumsum(is_new_E) - 1

# ---------- 2. 磁場チャンク ----------------------------------
t_ref_B = pt.data_quants['B16_gsm_x'].time.values
is_new_B = np.concatenate(([True], np.diff(t_ref_B) > gap_thr))
chunk_B  = np.cumsum(is_new_B) - 1

# ---------- 3. 分割ループ ------------------------------------
vars_E = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z']
vars_B = ['B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z']

def split_and_store(var_list, t_ref, chunk_id):
    for v in var_list:
        da   = pt.data_quants[v]
        for k in np.unique(chunk_id):
            sel = chunk_id == k
            if sel.sum() == 0:
                continue
            new = f'{v}_seg{k}'
            pt.store_data(new,
                          data={'x': t_ref[sel], 'y': da.values[sel]})
            unit = '[mV/m]' if v.startswith('e_') else '[nT]'
            pt.options(new,'ytitle',v); pt.options(new,'ysubtitle',unit)

split_and_store(vars_E, t_ref_E, chunk_E)
split_and_store(vars_B, t_ref_B, chunk_B)

# ---------- 4. まとめ変数 ------------------------------------
def make_all(base):
    pt.store_data(f'{base}_all', data=pt.tnames(f'{base}_seg*'))

for base in vars_E + vars_B:
    make_all(base)

# ---------- 5. 描画 ------------------------------------------
pt.options('E8_gsm_x_all', 'ytitle', 'E_x (gsm)')
pt.options('E8_gsm_y_all', 'ytitle', 'E_y (gsm)')
pt.options('E8_gsm_z_all', 'ytitle', 'E_z (gsm)')
pt.options(['E8_gsm_x_all', 'E8_gsm_y_all', 'E8_gsm_z_all'], 'ysubtitle', '[mV/m]')
pt.options('B16_gsm_x_all', 'ytitle', 'B_x (gsm)')
pt.options('B16_gsm_y_all', 'ytitle', 'B_y (gsm)')
pt.options('B16_gsm_z_all', 'ytitle', 'B_z (gsm)')
pt.options(['B16_gsm_x_all', 'B16_gsm_y_all', 'B16_gsm_z_all'], 'ysubtitle', '[nT]')

vars_to_plot = ['E8_gsm_x_all','E8_gsm_y_all','E8_gsm_z_all', 'B16_gsm_x_all','B16_gsm_y_all','B16_gsm_z_all']

if os.path.isdir(path_base_save_plot):
    pt.timespan('2022-09-01/20:00:00', 3, keyword='hour')
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_gsm_full.png')
    )
    plt.close(fig)
    pt.timespan('2022-09-01/20:45:00', 95, keyword='minute')
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_gsm_quiet.png')
    )
    plt.close(fig)
    pt.timespan('2022-09-01/22:25:00', 1, keyword='hour')
    # フォルダが存在 → プロットを「表示せずに保存」
    pt.options('B16_gsm_x_all', 'yrange', [-10, 10])
    pt.options('B16_gsm_y_all', 'yrange', [-10, 10])
    pt.options('B16_gsm_z_all', 'yrange', [0, 50])
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_gsm_KAW.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        vars_to_plot,
        display=True   # 省略可（デフォルト）
    )

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pandas as pd
import sys
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
import importlib
importlib.reload(pp)

# ────────────────────────────────────────────────
# 0. セグメント名リストを取得
# ────────────────────────────────────────────────
seg_ids = sorted({v.split('_seg')[-1]                      # {'0','1',...}
                  for v in pt.tnames('E8_gsm_x_seg*')})

# まとめ用リスト（後で overplot するために使う）
ex_all, ey_all, ez_all = [], [], []
bx_all, by_all, bz_all = [], [], []

seg_ids_B = sorted({v.split('_seg')[-1]                      # {'0','1',...}
                  for v in pt.tnames('B16_gsm_x_seg*')})

B_x_concat = pp.concat_tplot_segments('B16_gsm_x', seg_ids_B)
B_y_concat = pp.concat_tplot_segments('B16_gsm_y', seg_ids_B)
B_z_concat = pp.concat_tplot_segments('B16_gsm_z', seg_ids_B)

ds_B = xr.Dataset({'B16_gsm_x': B_x_concat, 'B16_gsm_y': B_y_concat, 'B16_gsm_z': B_z_concat}, coords={'time': B_x_concat.time})

print(ds_B)

for sid in seg_ids:
    # TVar 取り出し → pandas DataFrame へ （xarray に変換するため）
    def to_df(var):           # var = 'e_per1_seg0' など
        dq = pt.data_quants[var]
        return pd.DataFrame({'time': dq.time.values,
                             var.split('_seg')[0]: dq.values})

    df_E = (to_df(f'E8_gsm_x_seg{sid}')
            .merge(to_df(f'E8_gsm_y_seg{sid}'), on='time')
            .merge(to_df(f'E8_gsm_z_seg{sid}'),  on='time'))

    # ---- xarray Dataset へ ----
    ds_E = xr.Dataset({k:(('time',), df_E[k].to_numpy(dtype=float))
                       for k in ['E8_gsm_x','E8_gsm_y','E8_gsm_z']},
                      coords={'time': df_E['time'].to_numpy('datetime64[ns]')})

    rename_dict_B = {
        'B16_gsm_x': 'B8_gsm_x',
        'B16_gsm_y': 'B8_gsm_y',
        'B16_gsm_z': 'B8_gsm_z'
    }

    # renameメソッドで変数名を変更 (16 Hz -> 8 Hz)
    ds_B_16 = ds_B.rename(rename_dict_B)

    # ---- B を E の時刻へ線形補間 ----
    ds_B_8  = ds_B_16.interp(time=ds_E.time, method='linear')
    ds_merged = xr.merge([ds_E, ds_B_8])
    ds_merged = ds_merged.dropna(dim='time', how='any', subset=['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B8_gsm_x', 'B8_gsm_y', 'B8_gsm_z'])

    # ---- TVar 登録（seg ID を引き継ぐ） ----
    for var in ds_merged.data_vars:
        new_name = f'{var}_i_seg{sid}'
        pt.store_data(new_name,
                      data={'x': ds_merged.time.values,
                            'y': ds_merged[var].values})
        unit = '[mV/m]' if var.startswith('E8_') else '[nT]'
        pt.options(new_name, 'ytitle', var)
        pt.options(new_name, 'ysubtitle', unit)

        # まとめリストへ追加
        if   var == 'E8_gsm_x': ex_all.append(new_name)
        elif var == 'E8_gsm_y': ey_all.append(new_name)
        elif var == 'E8_gsm_z': ez_all.append(new_name)
        elif var == 'B8_gsm_x': bx_all.append(new_name)
        elif var == 'B8_gsm_y': by_all.append(new_name)
        elif var == 'B8_gsm_z': bz_all.append(new_name)

# ────────────────────────────────────────────────
# 2. overplot 用の “まとめ変数” を 6 つ作成
# ────────────────────────────────────────────────
pt.store_data('E8_gsm_x_all_i', data=ex_all)
pt.store_data('E8_gsm_y_all_i', data=ey_all)
pt.store_data('E8_gsm_z_all_i', data=ez_all)
pt.store_data('B8_gsm_x_all_i', data=bx_all)
pt.store_data('B8_gsm_y_all_i', data=by_all)
pt.store_data('B8_gsm_z_all_i', data=bz_all)

pt.options('E8_gsm_x_all_i', 'ytitle', 'E_x (gsm)')
pt.options('E8_gsm_y_all_i', 'ytitle', 'E_y (gsm)')
pt.options('E8_gsm_z_all_i', 'ytitle', 'E_z (gsm)')
pt.options(['E8_gsm_x_all_i', 'E8_gsm_y_all_i', 'E8_gsm_z_all_i'], 'ysubtitle', '[mV/m]')
pt.options('B8_gsm_x_all_i', 'ytitle', 'B_x (gsm)')
pt.options('B8_gsm_y_all_i', 'ytitle', 'B_y (gsm)')
pt.options('B8_gsm_z_all_i', 'ytitle', 'B_z (gsm)')
pt.options(['B8_gsm_x_all_i', 'B8_gsm_y_all_i', 'B8_gsm_z_all_i'], 'ysubtitle', '[nT]')

# ────────────────────────────────────────────────
# 3. プロット（6 パネル、各パネルに全セグメントが重なって表示）
# ────────────────────────────────────────────────

vars_to_plot = ['E8_gsm_x_all_i', 'E8_gsm_y_all_i', 'E8_gsm_z_all_i', 'B8_gsm_x_all_i', 'B8_gsm_y_all_i', 'B8_gsm_z_all_i']
if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    pt.options('B8_gsm_x_all_i', 'yrange', [-10, 10])
    pt.options('B8_gsm_y_all_i', 'yrange', [-10, 10])
    pt.options('B8_gsm_z_all_i', 'yrange', [0, 50])
    fig, axes = pt.tplot(
        vars_to_plot,
        display=True,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_gsm_interp.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        vars_to_plot,
        display=True   # 省略可（デフォルト）
    )

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pyspedas as psp
import matplotlib.pyplot as plt
import os
import pandas as pd

# ------------------------------------------------------------
# 補助関数：単一セグメントのFAC変換処理をまとめる
# ------------------------------------------------------------
def rotate_segment_to_fac(sid, vec_base_name, matrix_da):
    """
    指定されたセグメントのベクトルデータをFACへ変換し、成分に分割する。
    戻り値: 3つの成分変数名のリスト [comp_x, comp_y, comp_z]
    """
    print(f"  - Rotating vector: {vec_base_name}_i_seg{sid}")
    
    # 1. ベクトルデータを取得
    vec_tvar = f'{vec_base_name}_i_seg{sid}'
    t_vec, d_vec = pt.get_data(vec_tvar)

    # 1a. pandasを使って時間軸の重複をチェックし、削除する
    #     重複があった場合、最初の値を採用する (keep='first')
    unique_indices = pd.Index(t_vec).is_unique
    if not unique_indices:
        print(f"    - 警告: {vec_tvar} の時間軸に重複が見つかりました。重複を削除します。")
        _, unique_idx = np.unique(t_vec, return_index=True)
        t_vec = t_vec[unique_idx]
        d_vec = d_vec[unique_idx]

    # 2. 回転行列をベクトルの時間軸に手動で補間
    matrix_interp = matrix_da.interp(time_mat=t_vec, method="linear").values

    # 3. NumPyのeinsumで手動でベクトル回転
    fac_np = np.einsum('tij,tj->ti', matrix_interp, d_vec)
    
    prefix = 'E' if 'E' in vec_base_name else 'B'
    fac_tvar = f'{prefix}_fac_vec_seg{sid}'
    pt.store_data(fac_tvar, data={'x': t_vec, 'y': fac_np})
    
    default_component_names = psp.split_vec(fac_tvar) 
    desired_component_names = [f'{prefix}{c}_fac_seg{sid}' for c in ['x', 'y', 'z']]
    for i in range(3):
        pt.tplot_rename(default_component_names[i], desired_component_names[i])
    
    return desired_component_names

# ------------------------------------------------------------
# 0. パラメータ設定
# ------------------------------------------------------------
b_field_lf_tvar = 'tha_fgs_gsm'
e_field_hf_base = 'E8_gsm'
b_field_hf_base = 'B8_gsm'
seg_ids = sorted({v.split('_seg')[-1] for v in pt.tnames(f'{e_field_hf_base}_x_i_seg*')})

# 100 sec rolling
B_rolling = pt.data_quants[b_field_lf_tvar]
dt = (B_rolling.time.values[1] - B_rolling.time.values[0]).astype('timedelta64[ns]').astype(float) * 1e-9
win_secs = 100.0
win_pts = int(win_secs / dt)
B_rolling = B_rolling.rolling(time=win_pts, center=True).mean()
b_field_lf_tvar_rolling = b_field_lf_tvar + '_rolling'
print(b_field_lf_tvar_rolling)
pt.store_data(b_field_lf_tvar_rolling, data={'x': B_rolling.time, 'y': B_rolling.values}, attr_dict=B_rolling.attrs)

if os.path.isdir(path_base_save_plot):
    save_path = os.path.join(path_base_save_plot, 'B_rolling.png')
    fig, axes = pt.tplot(
        b_field_lf_tvar_rolling,
        display=False,
        return_plot_objects=True,
        save_png=save_path
    )
    plt.close(fig)
    print(f"Saved plot to {save_path}")
else:
    pt.tplot(b_field_lf_tvar_rolling, display=True)

# ------------------------------------------------------------
# 1. 成分データをベクトルに結合し、メタデータを設定
# ------------------------------------------------------------
print("--- ステップ1: 成分データをベクトルに結合し、座標系メタデータを設定 ---")
for sid in seg_ids:
    e_components = [f'{e_field_hf_base}_x_i_seg{sid}', f'{e_field_hf_base}_y_i_seg{sid}', f'{e_field_hf_base}_z_i_seg{sid}']
    e_vec_tvar = f'{e_field_hf_base}_i_seg{sid}'
    pt.join_vec(e_components, newname=e_vec_tvar)
    pt.data_quants[e_vec_tvar].attrs['coordinate_system'] = 'gsm'
    
    b_components = [f'{b_field_hf_base}_x_i_seg{sid}', f'{b_field_hf_base}_y_i_seg{sid}', f'{b_field_hf_base}_z_i_seg{sid}']
    b_vec_tvar = f'{b_field_hf_base}_i_seg{sid}'
    pt.join_vec(b_components, newname=b_vec_tvar)
    pt.data_quants[b_vec_tvar].attrs['coordinate_system'] = 'gsm'
print("ベクトル変数の作成とメタデータの設定が完了しました。")

# ------------------------------------------------------------
# 2. FAC回転行列の作成と準備
# ------------------------------------------------------------
print("\n--- ステップ2: FAC回転行列を作成し、準備 ---")
fac_matrix_tvar = psp.fac_matrix_make(b_field_lf_tvar_rolling)

t_mat, d_mat = pt.get_data(fac_matrix_tvar)

if t_mat is not None:
    unique_indices_mat = pd.Index(t_mat).is_unique
    if not unique_indices_mat:
        print(f"警告: 回転行列 ({fac_matrix_tvar}) の時間軸に重複が見つかりました。重複を削除します。")
        _, unique_idx = np.unique(t_mat, return_index=True)
        t_mat = t_mat[unique_idx]
        # d_matも同じインデックスでスライスして、時間とデータの整合性を保つ
        d_mat = d_mat[unique_idx]

if d_mat is not None and d_mat.ndim == 2:
    d_mat = d_mat[np.newaxis, :, :]
matrix_da = xr.DataArray(d_mat, dims=('time_mat', 'row', 'col'), coords={'time_mat': t_mat})
print(f"回転行列をxarray.DataArrayとして準備完了。Shape: {matrix_da.shape}")

# ------------------------------------------------------------
# 3. segment ごとに FAC 変換 (補助関数を利用)
# ------------------------------------------------------------
print("\n--- ステップ3: 各セグメントをFACへ変換 ---")
fac_vars = {'Ex': [], 'Ey': [], 'Ez': [], 'Bx': [], 'By': [], 'Bz': []}

for sid in seg_ids:
    print(f"\n--- Processing segment {sid} ---")
    
    # 補助関数を呼び出して電場と磁場をそれぞれ変換
    e_fac_components = rotate_segment_to_fac(sid, e_field_hf_base, matrix_da)
    b_fac_components = rotate_segment_to_fac(sid, b_field_hf_base, matrix_da)
    
    # 結果をリストに格納
    fac_vars['Ex'].append(e_fac_components[0])
    fac_vars['Ey'].append(e_fac_components[1])
    fac_vars['Ez'].append(e_fac_components[2])
    fac_vars['Bx'].append(b_fac_components[0])
    fac_vars['By'].append(b_fac_components[1])
    fac_vars['Bz'].append(b_fac_components[2])

# ------------------------------------------------------------
# 3. まとめ変数を作って overplot
# ------------------------------------------------------------
pt.store_data('Ex_fac_all', data=fac_vars['Ex'])
pt.store_data('Ey_fac_all', data=fac_vars['Ey'])
pt.store_data('Ez_fac_all', data=fac_vars['Ez'])
pt.store_data('Bx_fac_all', data=fac_vars['Bx'])
pt.store_data('By_fac_all', data=fac_vars['By'])
pt.store_data('Bz_fac_all', data=fac_vars['Bz'])

# オプション設定
pt.options('Ex_fac_all', 'ytitle', 'E_x (fac)')
pt.options('Ey_fac_all', 'ytitle', 'E_y (fac)')
pt.options('Ez_fac_all', 'ytitle', 'E_z (fac)')
pt.options(['Ex_fac_all', 'Ey_fac_all', 'Ez_fac_all'], 'ysubtitle', '[mV/m]')
pt.options('Bx_fac_all', 'ytitle', 'B_x (fac)')
pt.options('By_fac_all', 'ytitle', 'B_y (fac)')
pt.options('Bz_fac_all', 'ytitle', 'B_z (fac)')
pt.options(['Bx_fac_all', 'By_fac_all', 'Bz_fac_all'], 'ysubtitle', '[nT]')


# ------------------------------------------------------------
# 4. プロット
# ------------------------------------------------------------
vars_to_plot = ['Ex_fac_all','Ey_fac_all','Ez_fac_all', 'Bx_fac_all','By_fac_all','Bz_fac_all']


if os.path.isdir(path_base_save_plot):
    # --- フォルダが存在する場合：表示せずに保存 ---
    
    print("Generating plot objects...")
    pt.timespan('2022-09-01/20:00:00', 3, keyword='hour')
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,                # Jupyter上での自動表示を抑制
        return_plot_objects=True      # fig, axes を返してもらう
    )
    save_path = os.path.join(path_base_save_plot, 'EB_fields_fac_full.png')
    print(f"Saving plot to: {save_path}")
    fig.savefig(save_path, dpi=200) # dpiもここで指定できる
    plt.close(fig)

    pt.timespan('2022-09-01/20:45:00', 95, keyword='minute')
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,                # Jupyter上での自動表示を抑制
        return_plot_objects=True      # fig, axes を返してもらう
    )
    save_path = os.path.join(path_base_save_plot, 'EB_fields_fac_quiet.png')
    print(f"Saving plot to: {save_path}")
    fig.savefig(save_path, dpi=200) # dpiもここで指定できる
    plt.close(fig)

    pt.timespan('2022-09-01/22:25:00', 1, keyword='hour')
    pt.options('Bx_fac_all', 'yrange', [-10, 10])
    pt.options('By_fac_all', 'yrange', [-10, 10])
    pt.options('Bz_fac_all', 'yrange', [0, 50])
    fig, axes = pt.tplot(
        vars_to_plot,
        display=True,                # Jupyter上での自動表示を抑制
        return_plot_objects=True      # fig, axes を返してもらう
    )
    save_path = os.path.join(path_base_save_plot, 'EB_fields_fac_KAW.png')
    print(f"Saving plot to: {save_path}")
    fig.savefig(save_path, dpi=200) # dpiもここで指定できる
    plt.close(fig)
    print("Plotting complete and memory released.")

else:
    # --- フォルダが存在しない場合：通常表示 ---
    pt.tplot(
        vars_to_plot,
        display=True
    )

In [ ]:
import pytplot as pt
import numpy as np
import os

# ------------------------------------------------------------
# 1. パラメータ設定
# ------------------------------------------------------------
# psp.fac_matrix_make が作成した回転行列のtplot変数名
fac_matrix_tvar = 'tha_fgs_gsm_rolling_fac_mat'

# ------------------------------------------------------------
# 2. tplot変数からデータを取得し、角度を計算
# ------------------------------------------------------------
try:
    t_mat, d_mat = pt.get_data(fac_matrix_tvar)

    if t_mat is not None:
        # d_mat の形状が (N, 3, 3) であることを想定
        e3_z = d_mat[:, 2, 2]
        angle_deg = np.degrees(
            np.arccos(np.clip(e3_z, -1.0, 1.0))
        )
        
        # ------------------------------------------------------------
        # 3. 計算結果を新しいtplot変数として格納し、オプションを設定
        # ------------------------------------------------------------
        angle_tvar = 'B0_z_angle'
        pt.store_data(angle_tvar, data={'x': t_mat, 'y': angle_deg})
        
        # --- プロットのスタイルを設定 ---
        pt.options(angle_tvar, 'title', 'Rotation angle between B0 and GSM z-axis')
        pt.options(angle_tvar, 'ytitle', '∠(B0, z_gsm)')
        pt.options(angle_tvar, 'ysubtitle', '[deg]')
        pt.options(angle_tvar, 'grid', True)

        # ------------------------------------------------------------
        # 4. 条件に応じてプロットまたは画像保存
        # ------------------------------------------------------------
        if os.path.isdir(path_base_save_plot):
            # --- フォルダが存在する場合：表示せずに保存 ---
            save_path = os.path.join(path_base_save_plot, 'rotation_angle.png')
            print(f"Plotting and saving to {save_path}...")
            
            pt.tplot(
                angle_tvar,
                save_png=save_path,
                display=False
            )
            plt.close('all')
            print("Done.")
            

        else:
            # --- フォルダが存在しない場合：通常表示 ---
            print("Displaying plot...")
            pt.tplot(angle_tvar)

except KeyError:
    print(f"tplot変数 '{fac_matrix_tvar}' が見つかりません。")
except Exception as e:
    print(f"エラーが発生しました: {e}")

# Band-Stop Filterの適用

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 8.0                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth（2-pole × 2-stage）
spin_tone = 1.0/3.0           # 減衰させたいスピン周期 [Hz] (THEMIS: 3 sec)
# 除去したい周波数帯域 [Hz]
low_lim = spin_tone * 1
high_lim = spin_tone * 3

lowcut = spin_tone - 0.05   # 例: スピントーンの-0.05 Hz
highcut = spin_tone*3 + 0.15  # 例: スピントーンの+0.05 Hz

# btypeを'bandstop'に、Wnに周波数帯域 [low, high] を指定
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandstop', output='sos', fs=fs)


# ------------------ インパルス応答 (変更なし) ------------------
n = 2048
delta = np.zeros(n)
delta[n//2] = 1

h = sosfiltfilt(sos, delta)
t = (np.arange(n) - n//2) / fs

# ------------------ 周波数応答 (変更なし) ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)
H_dbl = np.abs(H)**2

# ------------------ プロット (タイトルとハイライトを変更) ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse Response (Band-stop filter)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)| (double-pass)')

# 除去帯域を半透明のグレーで示す
axs[1].axvspan(low_lim, high_lim, color='gray', alpha=0.3, label=f'Stop band ({low_lim:.2f}-{high_lim:.2f} Hz)')
axs[1].axvline(spin_tone, color='purple', ls='--', label=rf'1 $\times$ Spin tone = {spin_tone:.3f} Hz')
axs[1].axvline(spin_tone*3, color='green', ls='--', label=rf'3 $\times$ Spin tone = {spin_tone*3:.3f} Hz')

axs[1].set_title('Magnitude Response (Band-stop filter)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20, 10)
axs[1].set_xlim(1E-1, fs/2)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく

# ------------------------------------------------------------
# 0. Butterworth BSF の係数（君が指定した設定）
# ------------------------------------------------------------
fs = 8.0                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth
spin_tone = 1.0/3.0           # 減衰させたいスピン周期 [Hz]

# 除去したい周波数帯域
lowcut = spin_tone - 0.05
highcut = spin_tone*3 + 0.15

# フィルタ係数を計算
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandstop', output='sos', fs=fs)


# ------------------------------------------------------------
# 汎用的なフィルタ適用関数
# ------------------------------------------------------------
def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

# ------------------------------------------------------------
# 1. セグメント ID を列挙
# ------------------------------------------------------------
# 元データは 'Ex_fac_seg*' などを想定
seg_ids = sorted({v.split('_seg')[-1] for v in pt.tnames('Ex_fac_seg*')})

# overplot まとめ変数用
Ex_bs, Ey_bs, Ez_bs = [], [], []
Bx_bs, By_bs, Bz_bs = [], [], []

# ------------------------------------------------------------
# 2. 各 seg で BSF → TVar 登録
# ------------------------------------------------------------
for sid in seg_ids:
    for base in ['Ex_fac', 'Ey_fac', 'Ez_fac', 'Bx_fac', 'By_fac', 'Bz_fac']:
        src = f'{base}_seg{sid}'
        if src not in pt.data_quants:
            continue
            
        da = pt.data_quants[src]
        filtered_data = apply_filter_segmented(da.values, sos)

        # 出力変数名に _bs_ (band-stop) を使用
        dst = f'{base}_bs_seg{sid}'
        pt.store_data(dst, data={'x': da.time.values, 'y': filtered_data})

        # 保存しておいて後で overplot 用リストへ
        if   base == 'Ex_fac': Ex_bs.append(dst)
        elif base == 'Ey_fac': Ey_bs.append(dst)
        elif base == 'Ez_fac': Ez_bs.append(dst)
        elif base == 'Bx_fac': Bx_bs.append(dst)
        elif base == 'By_fac': By_bs.append(dst)
        elif base == 'Bz_fac': Bz_bs.append(dst)

        unit = '[mV/m]' if base.startswith('E') else '[nT]'
        pt.options(dst, 'ytitle', base.replace('_fac',' (FAC, BSF)'))
        pt.options(dst, 'ysubtitle', unit)

# ------------------------------------------------------------
# 3. まとめ変数を作って overplot
# ------------------------------------------------------------
pt.store_data('Ex_fac_bs_all', data=Ex_bs)
pt.store_data('Ey_fac_bs_all', data=Ey_bs)
pt.store_data('Ez_fac_bs_all', data=Ez_bs)
pt.store_data('Bx_fac_bs_all', data=Bx_bs)
pt.store_data('By_fac_bs_all', data=By_bs)
pt.store_data('Bz_fac_bs_all', data=Bz_bs)

# ------------------------------------------------------------
# 4. 描画（6 パネル）
# ------------------------------------------------------------
vars_bs = ['Ex_fac_bs_all', 'Ey_fac_bs_all', 'Ez_fac_bs_all',
           'Bx_fac_bs_all', 'By_fac_bs_all', 'Bz_fac_bs_all']

# path_base_save_plot は事前に定義されているとする
if 'path_base_save_plot' in locals() and os.path.isdir(path_base_save_plot):
    # 存在する → 表示せずに保存のみ
    # (tplotのTypeErrorを回避するため、保存処理を修正)
    pt.options('Bx_fac_bs_all', 'yrange', [-10, 10])
    pt.options('By_fac_bs_all', 'yrange', [-10, 10])
    pt.options('Bz_fac_bs_all', 'yrange', [0, 50])
    fig, axes = pt.tplot(vars_bs, display=True, return_plot_objects=True)
    save_path = os.path.join(path_base_save_plot, 'EB_fields_fac_bsf.png')
    fig.savefig(save_path, dpi=200)
    plt.close(fig)
    print(f"Plot saved to {save_path}")
else:
    # 存在しない → 通常表示
    pt.tplot(vars_bs)

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt

sys.path.append("..")
import module_handmade.tdwavelet as tw
importlib.reload(tw)

# ── 1. 各セグメントに対してウェーブレット計算とオプション設定 ───────────

# 各成分のtplot変数名を保存するためのリストを初期化
Ex_cwt, Ey_cwt, Ez_cwt = [], [], []
Bx_cwt, By_cwt, Bz_cwt = [], [], []

for sid in seg_ids:
    for EB, unit in [('E', '(mV/m)^2/Hz'), ('B', 'nT^2/Hz')]:
        for comp in ['x', 'y', 'z']:
            # セグメントIDを含む入力変数名を作成
            var_in = f'{EB}{comp}_fac_bs_seg{sid}'
            if var_in not in pt.data_quants:
                continue

            # ウェーブレット解析を実行
            # dt(サンプリング周期)はデータに合わせて要調整。ここでは仮に1/128秒としている。
            tw.tdwavelet(
                var_in,
                dt=1/8,
                s0=1/8*2E0,
                dj=1/16,
                suffix='_cwt',
                zrange=[1e-6, 1e3]
            )

            # 出力変数名
            old_name = f'{var_in}_cwt'
            var_out = f'{EB}{comp}_fac_bs_cwt_seg{sid}'
            pt.tplot_rename(old_name, var_out)

            # 軸ラベルなどの設定
            pt.options(var_out, 'ylog', 1)
            pt.options(var_out, 'zlog', 1)
            pt.options(var_out, 'ztitle', 'PSD')
            pt.options(var_out, 'zsubtitle', unit)
            pt.options(var_out, 'ytitle', f'{EB}_{comp} (FAC)')
            pt.options(var_out, 'ysubtitle', '[Hz]')
            pt.options(var_out, 'colormap', 'turbo')
            pt.options(var_out, 'zrange', [1e-6, 1e3]) # z軸の範囲を統一
            #pt.options(var_out, 'yrange', [0.45, 1E2])

            # 成分ごとにリストへ追加
            if   EB == 'E' and comp == 'x': Ex_cwt.append(var_out)
            elif EB == 'E' and comp == 'y': Ey_cwt.append(var_out)
            elif EB == 'E' and comp == 'z': Ez_cwt.append(var_out)
            elif EB == 'B' and comp == 'x': Bx_cwt.append(var_out)
            elif EB == 'B' and comp == 'y': By_cwt.append(var_out)
            elif EB == 'B' and comp == 'z': Bz_cwt.append(var_out)

# ── 2. セグメントを結合したリンク変数を作成 ─────────────────
# store_dataを使い、セグメント化されたスペクトルを一つにまとめる
pt.store_data('Ex_cwt_all', data=Ex_cwt)
pt.store_data('Ey_cwt_all', data=Ey_cwt)
pt.store_data('Ez_cwt_all', data=Ez_cwt)
pt.store_data('Bx_cwt_all', data=Bx_cwt)
pt.store_data('By_cwt_all', data=By_cwt)
pt.store_data('Bz_cwt_all', data=Bz_cwt)

In [ ]:
import sys
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
importlib.reload(pp)
import numpy as np

# ── 3. タイムスパンを変えつつプロット／保存 ─────────────────
time_windows = [
    np.datetime64('2022-09-01T20:45') + np.timedelta64(5, 'm')*n
    for n in range(32)
]

# プロット対象を「結合後」の変数リストに変更
vars_to_plot = [
    'Ex_fac_bs_cwt', 'Ey_fac_bs_cwt', 'Ez_fac_bs_cwt',
    'Bx_fac_bs_cwt', 'By_fac_bs_cwt', 'Bz_fac_bs_cwt'
]

vars_to_plot_all_c = [
    'Ex_fac_bs_cwt_all_c', 'Ey_fac_bs_cwt_all_c', 'Ez_fac_bs_cwt_all_c',
    'Bx_fac_bs_cwt_all_c', 'By_fac_bs_cwt_all_c', 'Bz_fac_bs_cwt_all_c'
]

for var in vars_to_plot:
    data = pp.concat_tplot_segments(var, seg_ids)
    pt.store_data(f'{var}_all_c', data={'x': data.time, 'y': data.data, 'v': data.spec_bins})

#for t0 in time_windows:
#    start_str = str(t0)
#    pt.timespan(start_str, 5, keyword='minute')
#    print(f"Processing window: {start_str}")
#    
#    # オプション設定
#    for EB, unit in [('B', 'nT^2/Hz'), ('E', '(mV/m)^2/Hz')]:
#        for comp in ['x', 'y', 'z']:
#            var_in = f'{EB}{comp}_fac_bs_cwt_all_c'
#            if var_in not in pt.data_quants:
#                print(f'skip (no {var_in})')
#                continue
#            pt.options(var_in, 'spec', True)
#            pt.options(var_in, 'ylog', 1)
#            pt.options(var_in, 'zlog', 1)
#            pt.options(var_in, 'ztitle', 'PSD')
#            pt.options(var_in, 'zsubtitle', unit)
#            pt.options(var_in, 'ytitle', f'{EB}_{comp} (FAC)')
#            pt.options(var_in, 'ysubtitle', '[Hz]')
#            pt.options(var_in, 'colormap', 'turbo')
#            pt.options(var_in, 'zrange', [1e-6, 1e3])
#            pt.options(var_in, 'yrange', [1e-2, 32])
#
#    # プロット・保存処理
#    if os.path.isdir(path_base_save_plot):
#        os.makedirs(path_base_save_plot, exist_ok=True)
#        fn_time = start_str.replace(':', '').replace('T', '_')
#        save_png = os.path.join(path_base_save_plot, f'EB_fields_fac_bs_cwt_{fn_time}.png')
#        
#        print(f"  -> Plotting and saving...")
#        fig, axes = pt.tplot(
#            vars_to_plot_all_c,
#            display=False,
#            return_plot_objects=True,
#            save_png=save_png
#        )
#        plt.close(fig)
#    else:
#        print(f"  -> Plotting...")
#        pt.tplot(vars_to_plot_all_c, display=True)

# 各成分のNoise(Median)を抽出

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pytplot as pt

# --- 1. PSD 変数リストと時間範囲 -----------------------------
vars_to_plot_all_c = [
    'Ex_fac_bs_cwt_all_c', 'Ey_fac_bs_cwt_all_c', 'Ez_fac_bs_cwt_all_c',
    'Bx_fac_bs_cwt_all_c', 'By_fac_bs_cwt_all_c', 'Bz_fac_bs_cwt_all_c'
]
t0, t1 = '2022-09-01T20:50:00', '2022-09-01T21:10:00'

# --- 2. 各変数ごとに median を計算して Dataset にまとめる ----
noise_da_dict = {}
for v in vars_to_plot_all_c:
    if v not in pt.data_quants:
        print(f'skip (no {v})')
        continue

    dq_cut = pt.data_quants[v].sel(time=slice(t0, t1))
    freq   = dq_cut.spec_bins.values
    med    = np.nanmedian(dq_cut.data, axis=0)

    med_da = xr.DataArray(
        data   = med,
        dims   = ['frequency'],
        coords = {'frequency': freq},
        name   = v
    )
    noise_da_dict[v] = med_da

noise_ds = xr.Dataset(noise_da_dict)
print(noise_ds)

# --- 3. プロット or 保存 --------------------------------------
fig, ax = plt.subplots(figsize=(6,4))
for key in noise_ds.data_vars:
    label = key.split('_')[0] + '_' + key.split('_')[2]
    ax.loglog(noise_ds['frequency'], noise_ds[key], label=label)

ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Median PSD')
ax.set_title(r'Noise floor 20:50-21:10 (median) (E: (mV/m)$^2$/Hz, B: (nT)$^2$/Hz)')
ax.grid(True, which='both', ls=':')
ax.legend()
ax.set_xlim(0.01, 32)
ax.set_ylim(1E-8, 5E1)
plt.tight_layout()

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → 保存のみ
    # ファイル名にコロンが入るとまずいので除去
    fn = f"noise_median_bs_{t0.replace(':','')}_{t1.replace(':','')}.png"
    save_path = os.path.join(path_base_save_plot, fn)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"saved to {save_path}")
else:
    # フォルダが存在しない → 通常表示
    plt.show()

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pytplot as pt

# -----------------------------------------------------------------
# 1. マスター周波数軸の事前作成
# -----------------------------------------------------------------
print("--- ステップ1: 全セグメントからマスター周波数軸を作成 ---")
all_cwt_vars = []
for sid in seg_ids:
    for EB in ['E', 'B']:
        for comp in ['x', 'y', 'z']:
            all_cwt_vars.append(f'{EB}{comp}_fac_bs_cwt_seg{sid}')

master_freq = np.array([], dtype=np.float64)
for var_name in all_cwt_vars:
    if var_name in pt.data_quants:
        da = pt.data_quants[var_name]
        if 'v' in da.coords:
            master_freq = np.union1d(master_freq, da.v.values)

print(f"マスター周波数軸を作成しました。サイズ: {len(master_freq)}")

# -----------------------------------------------------------------
# 2. ノイズフロアをマスター周波数軸に整列
# -----------------------------------------------------------------
# noise_dsは事前に計算済みと仮定
noise_ds_aligned = noise_ds.reindex({'frequency': master_freq})
print("ノイズフロアをマスター周波数軸に整列しました。")


# -----------------------------------------------------------------
# 3. メインループ：各セグメントのノイズ除去
# -----------------------------------------------------------------
print("\n--- ステップ3: 各セグメントのノイズ除去処理を開始 ---")
Ex_cwt, Ey_cwt, Ez_cwt = [], [], []
Bx_cwt, By_cwt, Bz_cwt = [], [], []

for sid in seg_ids:
    print(f"\n--- Processing segment {sid} ---")
    for EB, unit in [('E', '(mV/m)^2/Hz'), ('B', 'nT^2/Hz')]:
        for comp in ['x', 'y', 'z']:
            var_in = f'{EB}{comp}_fac_bs_cwt_seg{sid}'
            noise_var = f'{EB}{comp}_fac_bs_cwt_all_c'
            if var_in not in pt.data_quants:
                continue

            dq = pt.data_quants[var_in]
            
            # 最終的なデータを入れるための空の配列（NaNで満たされている）
            aligned_segment_data = np.full((len(dq.time), len(master_freq)), np.nan)
            
            # 共通の周波数とそのインデックスを取得
            _common, master_indices, segment_indices = np.intersect1d(
                master_freq, dq.v.values, return_indices=True
            )
            
            # 対応する列にデータをコピー
            aligned_segment_data[:, master_indices] = dq.data[:, segment_indices]
            # --- ▲▲▲▲▲ 手動整列ここまで ▲▲▲▲▲ ---

            # 整列済みのノイズベクトルを取得
            noise_vec = noise_ds_aligned[noise_var].values
            
            # 整列済みのデータからノイズを引く
            cleaned = aligned_segment_data - noise_vec[None, :]
            cleaned[cleaned <= 0] = np.nan

            # 出力変数名と格納
            var_out = f'{EB}{comp}_fac_bs_cwt_clean_seg{sid}'
            pt.store_data(
                var_out,
                data={'x': dq.time.values, 'y': cleaned, 'v': master_freq}
            )

            # 軸ラベルなどの設定
            pt.options(var_out, 'spec', True)
            pt.options(var_out, 'ylog', 1)
            pt.options(var_out, 'zlog', 1)
            pt.options(var_out, 'ztitle', 'PSD (S-N)')
            pt.options(var_out, 'zsubtitle', unit)
            pt.options(var_out, 'ytitle', f'{EB}_{comp} (FAC)')
            pt.options(var_out, 'ysubtitle', '[Hz]')
            pt.options(var_out, 'colormap', 'turbo')
            pt.options(var_out, 'zrange', [1e-6, 1e3])
            pt.options(var_out, 'yrange', [1e-2, 32])

            # 成分ごとにリストへ追加
            if   EB == 'E' and comp == 'x': Ex_cwt.append(var_out)
            elif EB == 'E' and comp == 'y': Ey_cwt.append(var_out)
            elif EB == 'E' and comp == 'z': Ez_cwt.append(var_out)
            elif EB == 'B' and comp == 'x': Bx_cwt.append(var_out)
            elif EB == 'B' and comp == 'y': By_cwt.append(var_out)
            elif EB == 'B' and comp == 'z': Bz_cwt.append(var_out)

pt.store_data('Ex_cwt_all_clean', data=Ex_cwt)
pt.store_data('Ey_cwt_all_clean', data=Ey_cwt)
pt.store_data('Ez_cwt_all_clean', data=Ez_cwt)
pt.store_data('Bx_cwt_all_clean', data=Bx_cwt)
pt.store_data('By_cwt_all_clean', data=By_cwt)
pt.store_data('Bz_cwt_all_clean', data=Bz_cwt)

In [ ]:
#import sys
#sys.path.append("..")
#import module_handmade.psd_plotter_themis as pp
#importlib.reload(pp)
#import numpy as np
#
## ── 3. タイムスパンを変えつつプロット／保存 ─────────────────
#time_windows = [
#    np.datetime64('2022-09-01T20:45') + np.timedelta64(5, 'm')*n
#    for n in range(32)
#]
#
#
## プロット対象を「結合後」の変数リストに変更
#vars_to_plot = [
#    'Ex_fac_bs_cwt_clean', 'Ey_fac_bs_cwt_clean', 'Ez_fac_bs_cwt_clean',
#    'Bx_fac_bs_cwt_clean', 'By_fac_bs_cwt_clean', 'Bz_fac_bs_cwt_clean'
#]
#
#vars_to_plot_all_c = [
#    'Ex_fac_bs_cwt_clean_all_c', 'Ey_fac_bs_cwt_clean_all_c', 'Ez_fac_bs_cwt_clean_all_c',
#    'Bx_fac_bs_cwt_clean_all_c', 'By_fac_bs_cwt_clean_all_c', 'Bz_fac_bs_cwt_clean_all_c'
#]
#
#for var in vars_to_plot:
#    data = pp.concat_tplot_segments(var, seg_ids)
#    pt.store_data(f'{var}_all_c', data={'x': data.time, 'y': data.data, 'v': data.spec_bins})
#
#for t0 in time_windows:
#    start_str = str(t0)
#    pt.timespan(start_str, 5, keyword='minute')
#    print(f"Processing window: {start_str}")
#    
#    # オプション設定
#    for EB, unit in [('B', 'nT^2/Hz'), ('E', '(mV/m)^2/Hz')]:
#        for comp in ['x', 'y', 'z']:
#            var_in = f'{EB}{comp}_fac_bs_cwt_clean_all_c'
#            if var_in not in pt.data_quants:
#                print(f'skip (no {var_in})')
#                continue
#            pt.options(var_in, 'spec', True)
#            pt.options(var_in, 'ylog', 1)
#            pt.options(var_in, 'zlog', 1)
#            pt.options(var_in, 'ztitle', 'PSD (S-N)')
#            pt.options(var_in, 'zsubtitle', unit)
#            pt.options(var_in, 'ytitle', f'{EB}_{comp} (FAC)')
#            pt.options(var_in, 'ysubtitle', '[Hz]')
#            pt.options(var_in, 'colormap', 'turbo')
#            pt.options(var_in, 'zrange', [1e-6, 1e3])
#            pt.options(var_in, 'yrange', [1e-2, 32])
#
#    # プロット・保存処理
#    if os.path.isdir(path_base_save_plot):
#        os.makedirs(path_base_save_plot, exist_ok=True)
#        fn_time = start_str.replace(':', '').replace('T', '_')
#        save_png = os.path.join(path_base_save_plot, f'EB_fields_fac_bs_cwt_clean_{fn_time}.png')
#        
#        print(f"  -> Plotting and saving...")
#        fig, axes = pt.tplot(
#            vars_to_plot_all_c,
#            display=False,
#            return_plot_objects=True,
#            save_png=save_png
#        )
#        plt.close(fig)
#    else:
#        print(f"  -> Plotting...")
#        pt.tplot(vars_to_plot_all_c, display=True)

# 軌道データから、衛星データ(GSM)を導出

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np

# 1. THEMIS-Aの軌道データをロード
psp.themis.state(
    probe='a',
    trange=time_range
)

vel_gsm   = pt.data_quants['tha_vel_gsm'].data   # (Nt,3)[R_E]
time_gsm = pt.data_quants['tha_vel_gsm'].time.values


time_gsm_s = time_gsm.astype('datetime64[ns]').astype(float) * 1e-9

pt.store_data('v_sc_gsm', data={'x': time_gsm_s, 'y': vel_gsm})
pt.options('v_sc_gsm', 'ytitle', r'$V_{\mathrm{sc}}$ (GSM) [km/s]')
pt.options('v_sc_gsm', 'legend_names', ['x','y','z'])

pt.timespan('2022-09-01/22:25:00', 1, keyword='hour')

if os.path.isdir(path_base_save_plot):
    png_path = os.path.join(path_base_save_plot, f'v_sc_gsm.png')
    pt.tplot('v_sc_gsm', save_png=png_path, display=False)
    plt.close('all')
    print(f"Saved: {png_path}")
else:
    pt.tplot('v_sc_gsm', display=True)

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pyspedas as psp # pyspedasもインポートしておく
import matplotlib.pyplot as plt
import os

# --- 0. パラメータ設定 ---
# 宇宙機速度と回転行列のtplot変数名
v_sc_gsm_tvar = 'v_sc_gsm' # この変数は事前にロードされている必要がある
fac_matrix_tvar = 'tha_fgs_gsm_rolling_fac_mat'

# --- 1. 回転行列データを準備 ---
try:
    t_mat, d_mat = pt.get_data(fac_matrix_tvar)

    if t_mat is not None:
        unique_indices_mat = pd.Index(t_mat).is_unique
        if not unique_indices_mat:
            print(f"警告: 回転行列 ({fac_matrix_tvar}) の時間軸に重複が見つかりました。重複を削除します。")
            _, unique_idx = np.unique(t_mat, return_index=True)
            t_mat = pd.to_datetime(t_mat[unique_idx].astype(float), unit='s')
            # d_matも同じインデックスでスライスして、時間とデータの整合性を保つ
            d_mat = d_mat[unique_idx]

    if d_mat is not None and d_mat.ndim == 2:
        d_mat = d_mat[np.newaxis, :, :]
    matrix_da = xr.DataArray(d_mat, dims=('time', 'row', 'col'), coords={'time': t_mat})
    print(f"回転行列をxarray.DataArrayとして準備完了。Shape: {matrix_da.shape}")

except KeyError:
    print(f"エラー: 回転行列'{fac_matrix_tvar}'が見つかりません。FAC変換スクリプトを先に実行してください。")
    exit()

# --- 2. 各セグメントをループして処理 ---
V_sc_fac_vars = []
V_sc_fac_perp_vars = []

for sid in seg_ids:
    # 各セグメントの基準となる時間軸を取得
    # （この変数は時間軸の参照にしか使わない）
    try:
        # .times でfloat配列を取得する代わりに、xarray=Trueでオブジェクト全体を取得
        psd_da = pt.get_data(f'Bx_fac_bs_cwt_clean_seg{sid}', xarray=True)
        # .time でdatetime64型を持つtime座標を取得する
        t_psd_datetime = psd_da.time
    except (KeyError, AttributeError):
        print(f"セグメント{sid}の時間軸が見つかりません。スキップします。")
        continue

    # 元の宇宙機速度データをセグメントの時間軸に補間
    V_sc_gsm_interp_da = pt.get_data(v_sc_gsm_tvar, xarray=True).interp(time=t_psd_datetime, method='linear')

    R_interp_np = matrix_da.interp(time=t_psd_datetime, method="linear").values
    V_sc_fac = np.einsum('tij,tj->ti', R_interp_np, V_sc_gsm_interp_da.values)

    V_sc_fac_perp = np.sqrt(V_sc_fac[:, 0]**2 + V_sc_fac[:, 1]**2)

    # --- tplotへの格納とオプション設定 ---
    new_name = f'v_sc_fac_seg{sid}'
    new_name_perp = f'v_sc_fac_perp_seg{sid}'
    V_sc_fac_vars.append(new_name)
    V_sc_fac_perp_vars.append(new_name_perp)

    pt.store_data(new_name, data={'x': t_psd_datetime, 'y': V_sc_fac})
    pt.store_data(new_name_perp, data={'x': t_psd_datetime, 'y': V_sc_fac_perp})

    
    pt.options(new_name, 'ytitle', r'$V_{\mathrm{sc}}$ (FAC)')
    if sid == seg_ids[-1]:
        pt.options(new_name, 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])
    else:
        pt.options(new_name, 'legend_names', None)
    pt.options(new_name, 'ysubtitle', r'[km/s]')
    pt.options(new_name_perp, 'ytitle', r'$V_{\mathrm{sc}\perp}$ (FAC)')
    pt.options(new_name_perp, 'ysubtitle', r'[km/s]')

# --- 3. まとめ変数とプロット (元のコードと同じ) ---
pt.store_data('v_sc_fac_all', data=V_sc_fac_vars)
pt.store_data('v_sc_fac_perp_all', data=V_sc_fac_perp_vars)
vars_to_plot = ['v_sc_fac_all', 'v_sc_fac_perp_all']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'v_sc_fac_and_perp.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.themis.mom(trange=time_range, probe='a', level='l2')

In [ ]:
import xarray as xr
import pytplot as pt
import numpy as np

ND_electron     = pt.data_quants['tha_peem_density']                # [/cc]
Temp_electron   = pt.data_quants['tha_peem_ptot'] / ND_electron     # [eV]

pt.store_data('Temp_electron', data={'x': Temp_electron.time, 'y': Temp_electron.data})

ND_ion          = pt.data_quants['tha_peim_density']                # [/cc]
Temp_ion        = pt.data_quants['tha_peim_ptot'] / ND_ion          # [eV]
Velocity_ion_gsm= pt.data_quants['tha_peim_velocity_gsm']           # [km/s]

pt.store_data('Temp_ion', data={'x': Temp_ion.time, 'y': Temp_ion.data})

print(ND_electron)
print(pt.data_quants['tha_peem_ptot'])
print(Temp_electron)
print(ND_ion)
print(Temp_ion)
print(Velocity_ion_gsm)

pt.options('tha_peem_density', 'ytitle', r'$n_{\mathrm{e}}$ [/cc]')
pt.options('Temp_electron', 'ytitle', r'$T_{\mathrm{e}}$ [eV]')

pt.options('tha_peim_density', 'ytitle', r'$n_{\mathrm{i}}$ [/cc]')
pt.options('Temp_ion', 'ytitle', r'$T_{\mathrm{i}}$ [eV]')
pt.options('tha_peim_velocity_gsm', 'ytitle', r'$V_{\mathrm{i}}$ (gsm) [km/s]')

vars_to_plot = ['tha_peem_density']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'electron_density.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

vars_to_plot = ['tha_peem_density', 'Temp_electron']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'electron_parameter.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

vars_to_plot = ['tha_peim_density', 'Temp_ion', 'tha_peim_velocity_gsm']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'ion_parameter.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

In [ ]:
import numpy as np
import xarray as xr
import pytplot as pt
import pyspedas as psp
import matplotlib.pyplot as plt
import os
import pandas as pd

# --- 0. パラメータ設定 ---
v_ion_gsm_tvar = 'tha_peim_velocity_gsm'
v_sc_gsm_tvar = 'v_sc_gsm'

fac_matrix_tvar = 'tha_fgs_gsm_rolling_fac_mat'

# --- 1. 回転行列データを準備 ---
try:
    t_mat, d_mat = pt.get_data(fac_matrix_tvar)

    if t_mat is not None:
        unique_indices_mat = pd.Index(t_mat).is_unique
        if not unique_indices_mat:
            print(f"警告: 回転行列 ({fac_matrix_tvar}) の時間軸に重複が見つかりました。重複を削除します。")
            _, unique_idx = np.unique(t_mat, return_index=True)
            t_mat = pd.to_datetime(t_mat[unique_idx].astype(float), unit='s')
            # d_matも同じインデックスでスライスして、時間とデータの整合性を保つ
            d_mat = d_mat[unique_idx]

    if d_mat is not None and d_mat.ndim == 2:
        d_mat = d_mat[np.newaxis, :, :]
    matrix_da = xr.DataArray(d_mat, dims=('time', 'row', 'col'), coords={'time': t_mat})
    print(f"回転行列をxarray.DataArrayとして準備完了。Shape: {matrix_da.shape}")

except KeyError:
    print(f"エラー: 回転行列'{fac_matrix_tvar}'が見つかりません。FAC変換スクリプトを先に実行してください。")
    exit()

# --- 2. 速度データを準備 ---
V_ion_gsm = pt.get_data(v_ion_gsm_tvar, xarray=True)
V_sc_gsm = pt.get_data(v_sc_gsm_tvar, xarray=True)

# --- 3. 各セグメントをループして処理 ---
v_ion_fac_vars, v_ion_fac_perp_vars = [], []
v_sys_fac_vars, v_sys_fac_perp_vars, v_sys_fac_perp_rolling_vars = [], [], []

for sid in seg_ids:
    try:
        t_psd = pt.get_data(f'Bx_fac_bs_cwt_clean_seg{sid}', xarray=True).time
    except (KeyError, AttributeError):
        print(f"セグメント{sid}の時間軸が見つかりません。スキップします。")
        continue

    # --- 物理量をセグメントの時間軸に補間 ---
    V_ion_gsm_interp = V_ion_gsm.interp(time=t_psd, method='linear').values
    V_sc_gsm_interp = V_sc_gsm.interp(time=t_psd, method='linear').values
    R_interp = matrix_da.interp(time=t_psd, method="linear").values

    # --- NumPyでFACへ変換 ---
    V_ion_fac = np.einsum('tij,tj->ti', R_interp, V_ion_gsm_interp)
    V_sc_fac = np.einsum('tij,tj->ti', R_interp, V_sc_gsm_interp)

    # --- 物理量の計算 ---
    V_sys_fac = V_ion_fac - V_sc_fac
    V_ion_fac_perp = np.sqrt(V_ion_fac[:, 0]**2 + V_ion_fac[:, 1]**2)
    V_sys_fac_perp = np.sqrt(V_sys_fac[:, 0]**2 + V_sys_fac[:, 1]**2)
    
    # --- tplotへの格納とオプション設定 ---
    # (イオン速度)
    new_name = f'v_ion_fac_seg{sid}'
    new_name_perp = f'v_ion_fac_perp_seg{sid}'
    v_ion_fac_vars.append(new_name)
    v_ion_fac_perp_vars.append(new_name_perp)

    pt.store_data(new_name, data={'x': t_psd, 'y': V_ion_fac})
    pt.store_data(new_name_perp, data={'x': t_psd, 'y': V_ion_fac_perp})

    pt.options(new_name, 'ytitle', r'$V_{\mathrm{i}}$ (FAC)')
    if sid == seg_ids[-1]:
        pt.options(new_name, 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])
    else:
        pt.options(new_name, 'legend_names', None)
    pt.options(new_name, 'ysubtitle', r'[km/s]')
    pt.options(new_name, 'char_size', 15)

    pt.options(new_name_perp, 'ytitle', r'$V_{\mathrm{i}\perp}$ (FAC)')
    pt.options(new_name_perp, 'char_size', 15)
    pt.options(new_name_perp, 'ysubtitle', r'[km/s]')

    # (システム速度)
    new_name_sys = f'v_sys_fac_seg{sid}'
    new_name_sys_perp = f'v_sys_fac_perp_seg{sid}'
    v_sys_fac_vars.append(new_name_sys)
    v_sys_fac_perp_vars.append(new_name_sys_perp)

    pt.store_data(new_name_sys, data={'x': t_psd, 'y': V_sys_fac})
    pt.store_data(new_name_sys_perp, data={'x': t_psd, 'y': V_sys_fac_perp})

    pt.options(new_name_sys, 'ytitle', r'$V_{\mathrm{sys}}$ (FAC)')
    if sid == seg_ids[-1]:
        pt.options(new_name_sys, 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])
    else:
        pt.options(new_name_sys, 'legend_names', None)
    pt.options(new_name_sys, 'ysubtitle', r'[km/s]')
    pt.options(new_name_sys, 'char_size', 15)

    pt.options(new_name_sys_perp, 'ytitle', r'$V_{\mathrm{sys}\perp}$ (FAC)')
    pt.options(new_name_sys_perp, 'char_size', 15)
    pt.options(new_name_sys_perp, 'ysubtitle', r'[km/s]')
    
    # (移動平均)
    temp_da = xr.DataArray(V_sys_fac_perp, dims=('time',), coords={'time': t_psd})
    dt_i = np.nanmedian(np.diff(t_psd.values)).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    V_sys_fac_perp_rolling = temp_da.rolling(time=win_pts_i, center=True).mean()
    
    new_name_sys_perp_rolling = f'v_sys_fac_perp_rolling_seg{sid}'
    v_sys_fac_perp_rolling_vars.append(new_name_sys_perp_rolling)
    
    pt.options(new_name_sys_perp_rolling, 'ytitle', r'$V_{\mathrm{sys}\perp}$')
    pt.options(new_name_sys_perp_rolling, 'ysubtitle', r'[km/s]')
    pt.options(new_name_sys_perp_rolling, 'char_size', 15)

    pt.store_data(new_name_sys_perp_rolling, data={'x': t_psd, 'y': V_sys_fac_perp_rolling.data})

# --- 4. まとめ変数とプロット ---
pt.store_data('v_ion_fac_all', data=v_ion_fac_vars)
pt.store_data('v_ion_fac_perp_all', data=v_ion_fac_perp_vars)
vars_to_plot_1 = ['v_ion_fac_all', 'v_ion_fac_perp_all']

pt.store_data('v_sys_fac_all', data=v_sys_fac_vars)
pt.store_data('v_sys_fac_perp_all', data=v_sys_fac_perp_vars)
vars_to_plot_2 = ['v_sys_fac_all', 'v_sys_fac_perp_all']

pt.store_data('v_sys_fac_perp_rolling_all', data=v_sys_fac_perp_rolling_vars)
pt.options('v_sys_fac_perp_rolling_all', 'ytitle', r'$V_{\mathrm{sys}\perp}$')
pt.options('v_sys_fac_perp_rolling_all', 'ysubtitle', r'[km/s]')
pt.options('v_sys_fac_perp_rolling_all', 'char_size', 15)

vars_to_plot_3 = ['v_sys_fac_perp_rolling_all']

# プロットを3回に分けて実行
if os.path.isdir(path_base_save_plot):
    save_png_1 = os.path.join(path_base_save_plot, 'v_ion_fac_and_perp.png')
    pt.tplot(vars_to_plot_1, display=False, save_png=save_png_1)
    print(f"Saved plot to {save_png_1}")

    save_png_2 = os.path.join(path_base_save_plot, 'v_sys_fac_and_perp.png')
    pt.tplot(vars_to_plot_2, display=False, save_png=save_png_2)
    print(f"Saved plot to {save_png_2}")
    
    save_png_3 = os.path.join(path_base_save_plot, 'v_sys_fac_perp_rolling.png')
    pt.tplot(vars_to_plot_3, display=False, save_png=save_png_3)
    print(f"Saved plot to {save_png_3}")
    
    plt.close('all')
else:
    print("--- V_ion plots ---")
    pt.tplot(vars_to_plot_1)
    print("\n--- V_sys plots ---")
    pt.tplot(vars_to_plot_2)
    print("\n--- V_sys_perp_rolling plots ---")
    pt.tplot(vars_to_plot_3)

In [ ]:
import xarray as xr
import pytplot as pt
import numpy as np
import pandas as pd

ion_acoustic_speed_vars, ion_thermal_speed_vars, Alfven_speed_vars, ion_cyclo_freq_vars, beta_ion_vars, tau_vars = [], [], [], [], [], []

proton_mass = 1.6726219e-27  # kg
elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

for sid in seg_ids:
    psd_var = f'Bx_fac_bs_cwt_clean_seg{sid}'

    try:
        psd_da = pt.data_quants[psd_var]
        t_psd_raw = psd_da.time.values # .valuesでnumpy配列を取得
        
        # pandas.uniqueを使って重複しない時間配列を取得
        # これによりt_psdは必ずユニークになる
        t_psd_unique = pd.unique(t_psd_raw)
        
        # もし重複があった場合は、データもユニークな時間に対応させる
        if len(t_psd_unique) < len(t_psd_raw):
            print(f"警告: セグメント {sid} の時間軸に重複が見つかりました。重複を削除します。")
            # uniqueな時間に対応するインデックスを取得し、データもスライスする
            _, unique_idx = np.unique(t_psd_raw, return_index=True)
            psd_da = psd_da[unique_idx]
            # 更新されたユニークな時間軸を再度取得
            t_psd = psd_da.time
        else:
            t_psd = psd_da.time

    except KeyError:
        print(f"警告: 変数 {psd_var} が見つかりません。セグメント {sid} をスキップします。")
        continue

    ion_acoustic_speed = np.sqrt(pt.data_quants['Temp_electron'] * elementary_charge / proton_mass).interp(time=t_psd, method='linear')
    dt_i = (ion_acoustic_speed.time[1] - ion_acoustic_speed.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    ion_acoustic_speed = ion_acoustic_speed.rolling(time=win_pts_i, center=True).mean('time')

    new_ion_acoustic = f'ion_acoustic_speed_seg{sid}'
    ion_acoustic_speed_vars.append(new_ion_acoustic)

    pt.store_data(new_ion_acoustic, data={'x': t_psd, 'y': ion_acoustic_speed*1E-3})
    pt.options(new_ion_acoustic, 'ytitle', r'$c_{\mathrm{s}}$')
    pt.options(new_ion_acoustic, 'ysubtitle', '[km/s]')


    ion_thermal_speed = np.sqrt(2E0 * pt.data_quants['Temp_ion'] * elementary_charge / proton_mass).interp(time=t_psd, method='linear')
    dt_i = (ion_thermal_speed.time[1] - ion_thermal_speed.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    ion_thermal_speed = ion_thermal_speed.rolling(time=win_pts_i, center=True).mean('time')

    new_ion_thermal = f'ion_thermal_speed_seg{sid}'
    ion_thermal_speed_vars.append(new_ion_thermal)

    pt.store_data(new_ion_thermal, data={'x': t_psd, 'y': ion_thermal_speed*1E-3})
    pt.options(new_ion_thermal, 'ytitle', r'$v_{\mathrm{thi}}$')
    pt.options(new_ion_thermal, 'ysubtitle', '[km/s]')

    print(pt.data_quants['tha_fgs_btotal'])
    print(t_psd)

    B_total = pt.data_quants['tha_fgs_btotal'].sortby('time')
    _, unique_idx = np.unique(B_total.time, return_index=True)
    B_total = B_total[unique_idx]
    B_total = B_total.interp(time=t_psd, method='linear')
    ND_electron_interp = ND_electron.interp(time=t_psd, method='linear')
    Alfven_speed = B_total*1E-9 / np.sqrt(mu0 * proton_mass * ND_electron_interp*1E6)
    dt_i = (Alfven_speed.time[1] - Alfven_speed.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    Alfven_speed = Alfven_speed.rolling(time=win_pts_i, center=True).mean('time')

    new_Alfven_speed = f'Alfven_speed_seg{sid}'
    Alfven_speed_vars.append(new_Alfven_speed)

    pt.store_data(new_Alfven_speed, data={'x': t_psd, 'y': Alfven_speed*1E-3})
    pt.options(new_Alfven_speed, 'ytitle', r'$v_{\mathrm{A}}$')
    pt.options(new_Alfven_speed, 'ysubtitle', '[km/s]')


    ion_cyclo_freq = elementary_charge * B_total*1E-9 / proton_mass / 2E0 / np.pi
    dt_i = (ion_cyclo_freq.time[1] - ion_cyclo_freq.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    ion_cyclo_freq = ion_cyclo_freq.rolling(time=win_pts_i, center=True).mean('time')

    new_ion_cyclo_freq = f'ion_cyclo_freq_seg{sid}'
    ion_cyclo_freq_vars.append(new_ion_cyclo_freq)

    pt.store_data(new_ion_cyclo_freq, data={'x': t_psd, 'y': ion_cyclo_freq})
    pt.options(new_ion_cyclo_freq, 'ytitle', r'$f_{\mathrm{ci}}$')
    pt.options(new_ion_cyclo_freq, 'ysubtitle', '[Hz]')

    
    beta_ion = (ion_thermal_speed / Alfven_speed)**2E0
    dt_i = (beta_ion.time[1] - beta_ion.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    beta_ion = beta_ion.rolling(time=win_pts_i, center=True).mean('time')

    new_beta_ion = f'beta_ion_seg{sid}'
    beta_ion_vars.append(new_beta_ion)

    pt.store_data(new_beta_ion, data={'x': t_psd, 'y': beta_ion})
    pt.options(new_beta_ion, 'ytitle', r'$\beta_{\mathrm{i}}$')
    
    tau = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0
    dt_i = (tau.time[1] - tau.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
    win_pts_i = int(100.0 / dt_i)
    tau = tau.rolling(time=win_pts_i, center=True).mean('time')

    new_tau = f'tau_seg{sid}'
    tau_vars.append(new_tau)

    pt.store_data(new_tau, data={'x': t_psd, 'y': tau})
    pt.options(new_tau, 'ytitle', r'$\tau$')

pt.store_data('ion_thermal_speed_all', data=ion_thermal_speed_vars)
pt.store_data('ion_acoustic_speed_all', data=ion_acoustic_speed_vars)
pt.store_data('Alfven_speed_all', data=Alfven_speed_vars)
pt.store_data('ion_cycl_freq_all', data=ion_cyclo_freq_vars)

pt.options('ion_thermal_speed_all', 'yrange', [500, 1000])
pt.options('ion_acoustic_speed_all', 'yrange', [100, 600])
pt.options('Alfven_speed_all', 'yrange', [500, 1500])
pt.options('ion_cycl_freq_all', 'yrange', [0.3, 0.6])


vars_to_plot = ['ion_thermal_speed_all', 'ion_acoustic_speed_all', 'Alfven_speed_all', 'ion_cycl_freq_all']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_1.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )


pt.store_data('beta_ion_all', data=beta_ion_vars)
pt.store_data('tau_all', data=tau_vars)

pt.options('beta_ion_all', 'yrange', [0.2, 1.8])
pt.options('tau_all', 'yrange', [0.8, 20])
pt.options('tau_all', 'ylog', True)

vars_to_plot = ['beta_ion_all', 'tau_all']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_2.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_1sec_avg_low_freq'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（モジュール関数を呼び出す）
# ------------------------------------------------------------
data_dict = pp.load_and_prepare_data(cutoff_freq=[1/100, 1/3, 1])

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 解析対象の時間を決め、1秒ごとにループ
# ------------------------------------------------------------

def process_and_save_plot(t_start, data_dict, out_dir):
    """
    指定された単一の時刻について、スペクトルをプロットし、画像を保存する関数。
    """
    # モジュール関数を呼び出してプロットを作成
    # psd_plotter を pp としてインポートしている前提
    fig = pp.plot_freq_spectrum(t_start, data_dict, interval_sec=1)
    
    # figがNoneでなければ（データがあってプロットが作成されれば）保存
    if fig is not None:
        try:
            # ファイル名に使いやすいように文字列に変換
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=200)
        finally:
            # 保存に失敗しても、メモリ解放のために必ずクローズする
            plt.close(fig)

time_range = ['2022-09-01T20:45:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)

#process_and_save_plot(t_min, data_dict, out_dir)

# pandas.date_rangeで1秒ごとのタイムスタンプを生成
time_steps = pd.date_range(start=t_min, end=t_max, freq='1s')

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_plot)(t_start, data_dict, out_dir) for t_start in time_steps
)

print('Finished saving all plots!')

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
import gc # ガベージコレクションをインポート
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_120sec_avg_k_rhoi_low_freq_fit_3_43'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（ループ前に一度だけ実行）
# ------------------------------------------------------------
print("Loading and preparing data...")
data_dict = pp.load_and_prepare_data(cutoff_freq=[1/100, 1/3, 1])

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 並列処理のためのラッパー関数を定義
# ------------------------------------------------------------
def process_and_save_k_plot(t_start, data_dict, out_dir, k_range, n_bins, fit_range, dpi=200):
    """
    指定された単一の時刻について、kスペクトルをプロットし、画像を保存する関数。
    """
    plot_output = pp.plot_k_spectrum(
        t_start, 
        data_dict, 
        interval_sec=120,
        k_range=k_range, 
        n_bins=n_bins,
        fit_range=fit_range,
        n_samples_mc=200 # メモリ対策
    )
    
   # 戻り値がNoneでなければ、アンパックして処理を続ける
    if plot_output is not None:
        fig, fit_results = plot_output # ここでアンパック
        try:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S_k_spec.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=dpi)
        finally:
            plt.close(fig)
            gc.collect()
        return fit_results
    else:
        # plot_k_spectrumがNoneを返した場合、このワーカーもNoneを返す
        return None

# ------------------------------------------------------------
# 3. 解析対象の時間を決め、並列処理で一気に実行
# ------------------------------------------------------------
time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq='120s')

m_e = 9.1093837E-31
m_i = 1.67262192E-27
sqrt_m_i_m_e = np.sqrt(m_i / m_e)

# プロットのパラメータ
k_range_to_use      = (1e-1, 1e2)
n_bins_to_use       = 30
fit_range_to_use    = (3, sqrt_m_i_m_e)

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
results_list = Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_k_plot)(
        t_start, 
        data_dict, 
        out_dir, 
        k_range=k_range_to_use, 
        n_bins=n_bins_to_use,
        fit_range=fit_range_to_use
    ) for t_start in time_steps
)

print('Finished saving all plots!')

# ------------------------------------------------------------
# 4. フィッティング結果の保存と可視化
# ------------------------------------------------------------
print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    # --- 時間変化をプロット ---
    fig_kappa, ax = plt.subplots(figsize=(12, 5))
    
    # kappa_Bのプロット（エラーバー付き）
    ax.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_B$', ms=4, elinewidth=1, capsize=3)
    
    # kappa_Eのプロット（エラーバー付き）
    ax.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_E$', ms=4, elinewidth=1, capsize=3)
    
    ax.set_title('Time evolution of spectral index $\kappa$ (THEMIS-A)')
    ax.set_ylabel('Spectral index $\kappa$')
    ax.set_xlabel('Time')
    ax.legend()
    ax.minorticks_on()
    ax.grid(True, linestyle=':', which='both')
    
    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- 0. matplotlib 設定 ---
mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# --- 1. CSVファイルの読み込み ---
# CSVファイルをpandas DataFrameとして読み込む
time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)
time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
time_str_end = t_max.strftime('%Y%m%d_%H%M%S')

out_dir = '/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_15sec_avg_k_rhoi_low_freq'
csv_path = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
csv_path = os.path.join(out_dir, kappa_csv_filename)
try:
    df_kappa = pd.read_csv(csv_path)
    print("CSVファイルの読み込みに成功しました。")
    print("--- データフレームの最初の5行 ---")
    print(df_kappa.head())
except FileNotFoundError:
    print("エラー: 'kappa_timeseries.csv' が見つかりません。")
    exit()

# --- 2. データの前処理 ---
# 'time'列を文字列からdatetimeオブジェクトに変換し、データフレームのインデックスに設定
df_kappa['time'] = pd.to_datetime(df_kappa['time'])
df_kappa.set_index('time', inplace=True)


# --- 3. プロットの作成 ---
print("\nプロットを作成中...")
fig, ax = plt.subplots(figsize=(12, 5))

# kappa_Bのプロット（エラーバー付き）
# 欠損値(NaN)がある行は自動的にプロットから除外される
ax.errorbar(df_kappa.index, df_kappa['kappa_B'], yerr=df_kappa['kappa_B_err'],
            fmt='o-', color='blue', label=r'$\kappa_B$', ms=4, elinewidth=1, capsize=3)

# kappa_Eのプロット（エラーバー付き）
ax.errorbar(df_kappa.index, df_kappa['kappa_E'], yerr=df_kappa['kappa_E_err'],
            fmt='o-', color='orange', label=r'$\kappa_E$', ms=4, elinewidth=1, capsize=3)

# --- 4. プロットの装飾 ---
ax.set_title('Time evolution of spectral index $\kappa$')
ax.set_ylabel('Spectral index $\kappa$')
ax.set_xlabel('Time')
ax.legend()
ax.minorticks_on()
ax.grid(True, linestyle=':', which='both')

plt.tight_layout()
plt.show()

print("プロットが完了しました。")